In [3]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.data import DataLoader, TensorDataset

# ==============================
# SETTINGS
# ==============================

DATASET_ROOT = r"C:\Users\natra\OneDrive\Documents\cv project"

SEQ_LEN = 107
WINDOW_STRIDE = 5

BATCH_SIZE = 16
EPOCHS = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

yolo = YOLO("yolov5mu.pt")

# ==============================
# CBBA FEATURE EXTRACTION
# ==============================

def extract_cbba(video_path):

    cap = cv2.VideoCapture(video_path)

    areas = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = yolo(frame, classes=[2], verbose=False)

        if len(results[0].boxes) == 0:
            areas.append(np.nan)
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()

        areas_boxes = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        idx = np.argmax(areas_boxes)

        x1,y1,x2,y2 = boxes[idx]

        area = (x2-x1)*(y2-y1)

        areas.append(area)

    cap.release()

    areas = np.array(areas)

    valid = np.where(~np.isnan(areas))[0]

    if len(valid) < 20:
        return None

    start = valid[0]
    end = valid[-1]

    areas = areas[start:end+1]

    nans = np.isnan(areas)

    if np.any(nans):

        x = np.arange(len(areas))
        areas[nans] = np.interp(x[nans], x[~nans], areas[~nans])

    return areas


# ==============================
# CREATE SLIDING WINDOWS
# ==============================

def create_sequences(signal):

    sequences = []

    for i in range(0, len(signal) - SEQ_LEN, WINDOW_STRIDE):

        seq = signal[i:i+SEQ_LEN]
        sequences.append(seq)

    return sequences


# ==============================
# DATASET CREATION
# ==============================

X = []
y = []
vehicle_names = []

max_cbba = 0

print("Extracting CBBA features...")

for vehicle in os.listdir(DATASET_ROOT):

    vehicle_path = os.path.join(DATASET_ROOT, vehicle)

    if not os.path.isdir(vehicle_path):
        continue

    for file in os.listdir(vehicle_path):

        if not file.endswith(".MP4"):
            continue

        video_path = os.path.join(vehicle_path,file)
        txt_path = video_path.replace(".MP4",".txt")

        if not os.path.exists(txt_path):
            continue

        with open(txt_path) as f:
            speed,_ = map(float,f.readline().split())

        cbba = extract_cbba(video_path)

        if cbba is None:
            continue

        max_cbba = max(max_cbba, np.max(cbba))

        sequences = create_sequences(cbba)

        for seq in sequences:

            X.append(seq)
            y.append(speed)
            vehicle_names.append(vehicle)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# ==============================
# NORMALIZATION
# ==============================

X = X / max_cbba

mean_speed = np.mean(y)
std_speed = np.std(y)

y = (y - mean_speed) / std_speed

# ==============================
# MODEL (GRU)
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.gru = nn.GRU(
            input_size=1,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Sequential(

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,1)
        )

    def forward(self,x):

        x = x.squeeze(1).unsqueeze(-1)

        out,_ = self.gru(x)

        out = out[:,-1,:]

        return self.fc(out)


model = SpeedNet().to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = nn.SmoothL1Loss()

# ==============================
# VEHICLE-WISE TRAINING
# ==============================

unique_vehicles = list(set(vehicle_names))

all_preds = []
all_true = []

for test_vehicle in unique_vehicles:

    print("Testing vehicle:", test_vehicle)

    train_idx = [i for i,v in enumerate(vehicle_names) if v != test_vehicle]
    test_idx = [i for i,v in enumerate(vehicle_names) if v == test_vehicle]

    X_train = torch.tensor(X[train_idx]).float().unsqueeze(1).to(DEVICE)
    y_train = torch.tensor(y[train_idx]).float().to(DEVICE)

    X_test = torch.tensor(X[test_idx]).float().unsqueeze(1).to(DEVICE)
    y_test = torch.tensor(y[test_idx]).float().to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(X_train,y_train),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for xb,yb in train_loader:

            optimizer.zero_grad()

            preds = model(xb).squeeze()

            loss = criterion(preds,yb)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} Loss {total_loss/len(train_loader):.2f}")

    model.eval()

    with torch.no_grad():

        preds = model(X_test).cpu().numpy().flatten()

    true = y_test.cpu().numpy()

    all_preds.extend(preds)
    all_true.extend(true)

# ==============================
# FINAL METRICS
# ==============================

all_preds = np.array(all_preds)
all_true = np.array(all_true)

all_preds = all_preds * std_speed + mean_speed
all_true = all_true * std_speed + mean_speed

rmse = np.sqrt(mean_squared_error(all_true,all_preds))
mae = mean_absolute_error(all_true,all_preds)
r2 = r2_score(all_true,all_preds)

print("\nEvaluation Metrics")
print("-------------------")
print("RMSE :", rmse)
print("MAE  :", mae)
print("R2 Score :", r2)

# ==============================
# SAVE MODEL
# ==============================

torch.save({
    "model":model.state_dict(),
    "max_cbba":max_cbba,
    "mean_speed":mean_speed,
    "std_speed":std_speed
},"speed_model_cbba_gru.pth")

print("Model saved")

Device: cuda
Extracting CBBA features...
Dataset shape: (1843, 107)
Testing vehicle: KiaSportage
Epoch 0 Loss 0.46
Epoch 10 Loss 0.33
Epoch 20 Loss 0.24
Epoch 30 Loss 0.10
Epoch 40 Loss 0.04
Epoch 50 Loss 0.04
Epoch 60 Loss 0.03
Epoch 70 Loss 0.03
Epoch 80 Loss 0.02
Epoch 90 Loss 0.03
Testing vehicle: MercedesAMG550
Epoch 0 Loss 0.02
Epoch 10 Loss 0.02
Epoch 20 Loss 0.02
Epoch 30 Loss 0.02
Epoch 40 Loss 0.02
Epoch 50 Loss 0.01
Epoch 60 Loss 0.01
Epoch 70 Loss 0.02
Epoch 80 Loss 0.01
Epoch 90 Loss 0.01
Testing vehicle: CitroenC4Picasso
Epoch 0 Loss 0.03
Epoch 10 Loss 0.02
Epoch 20 Loss 0.02
Epoch 30 Loss 0.02
Epoch 40 Loss 0.02
Epoch 50 Loss 0.02
Epoch 60 Loss 0.02
Epoch 70 Loss 0.02
Epoch 80 Loss 0.01
Epoch 90 Loss 0.02
Testing vehicle: Mazda3
Epoch 0 Loss 0.02
Epoch 10 Loss 0.02
Epoch 20 Loss 0.01
Epoch 30 Loss 0.01
Epoch 40 Loss 0.01
Epoch 50 Loss 0.02
Epoch 60 Loss 0.01
Epoch 70 Loss 0.01
Epoch 80 Loss 0.01
Epoch 90 Loss 0.01
Testing vehicle: NissanQashqai
Epoch 0 Loss 0.02
Epoch 10

In [55]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\OpelInsignia\OpelInsignia\OpelInsignia_100.MP4"
OUTPUT_PATH = r"C:\Users\natra\Downloads\output_side_view.mp4"

SEQ_LEN = 107
WINDOW_STRIDE = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# MODEL ARCHITECTURE
# ==============================

class SpeedNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.gru = nn.GRU(
            input_size=1,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(self,x):

        x = x.squeeze(1).unsqueeze(-1)

        out,_ = self.gru(x)

        out = out[:,-1,:]

        return self.fc(out)

# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba_gru.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]
mean_speed = checkpoint["mean_speed"]
std_speed = checkpoint["std_speed"]

print("Model loaded successfully")

# ==============================
# YOLO MODEL
# ==============================

yolo = YOLO("yolov5mu.pt")

# ==============================
# CBBA EXTRACTION
# ==============================

def extract_cbba(video_path):

    cap = cv2.VideoCapture(video_path)

    areas = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = yolo(frame, classes=[2], verbose=False)

        if len(results[0].boxes) == 0:

            areas.append(np.nan)
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()

        box_areas = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        idx = np.argmax(box_areas)

        x1,y1,x2,y2 = boxes[idx]

        area = (x2-x1)*(y2-y1)

        areas.append(area)

    cap.release()

    areas = np.array(areas)

    valid = np.where(~np.isnan(areas))[0]

    if len(valid) < 20:
        return None

    start = valid[0]
    end = valid[-1]

    areas = areas[start:end+1]

    nans = np.isnan(areas)

    if np.any(nans):

        x = np.arange(len(areas))
        areas[nans] = np.interp(x[nans], x[~nans], areas[~nans])

    return areas

# ==============================
# CREATE WINDOWS
# ==============================

def create_sequences(signal):

    sequences = []

    for i in range(0, len(signal) - SEQ_LEN, WINDOW_STRIDE):

        seq = signal[i:i+SEQ_LEN]
        sequences.append(seq)

    return sequences

# ==============================
# SPEED PREDICTION
# ==============================

def predict_speed(video_path):

    cbba = extract_cbba(video_path)

    if cbba is None:
        print("Vehicle detection failed")
        return None

    sequences = create_sequences(cbba)

    preds = []

    for seq in sequences:

        seq = seq / max_cbba

        tensor = torch.tensor(seq).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

        with torch.no_grad():

            pred = model(tensor).item()

        preds.append(pred)

    preds = np.array(preds)

    preds = preds * std_speed + mean_speed

    final_speed = np.mean(preds)

    return final_speed

# ==============================
# RUN PREDICTION
# ==============================

speed = predict_speed(VIDEO_PATH)

if speed is not None:

    print("Predicted Speed:", round(speed,2), "km/h")

# ==============================
# DISPLAY + SAVE VIDEO
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = yolo(frame, classes=[2], verbose=False)

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes.xyxy.cpu().numpy()

        areas = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        idx = np.argmax(areas)

        x1,y1,x2,y2 = boxes[idx].astype(int)

        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

        cv2.putText(
            frame,
            f"{speed:.2f} km/h",
            (x1,y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0,0,255),
            2
        )

    # ✅ SAVE FRAME (only added)
    out.write(frame)

    cv2.imshow("Vehicle Speed Estimation", frame)

    if cv2.waitKey(25) & 0xFF == 27:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Output saved at:", OUTPUT_PATH)

Device: cuda
Model loaded successfully
Predicted Speed: 96.61 km/h
Output saved at: C:\Users\natra\Downloads\output_side_view.mp4


In [51]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\videoplayback.mp4"
OUTPUT_PATH = r"C:\Users\natra\Downloads\top_veiw.mp4"

SEQ_LEN = 107
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ==============================
# MODEL (GRU)
# ==============================

class SpeedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, 64, 2, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.squeeze(1).unsqueeze(-1)
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

# ==============================
# LOAD MODEL
# ==============================

checkpoint = torch.load("speed_model_cbba_gru.pth", map_location=DEVICE)

model = SpeedNet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

max_cbba = checkpoint["max_cbba"]
mean_speed = checkpoint["mean_speed"]
std_speed = checkpoint["std_speed"]

print("Model loaded successfully")

# ==============================
# YOLO
# ==============================

yolo = YOLO("yolov5mu.pt")
VEHICLE_CLASSES = [2, 3, 5, 7]

# ==============================
# TRACKING
# ==============================

tracks = {}
centers_prev = {}
speeds = {}
predicted = set()
next_id = 0

def assign_id(center):
    global next_id

    for vid, prev in centers_prev.items():
        if np.linalg.norm(np.array(center) - np.array(prev)) < 60:
            centers_prev[vid] = center
            return vid

    next_id += 1
    centers_prev[next_id] = center
    return next_id

# ==============================
# RESIZE CBBA
# ==============================

def resize_cbba(cbba):
    x_old = np.linspace(0, 1, len(cbba))
    x_new = np.linspace(0, 1, SEQ_LEN)
    return np.interp(x_new, x_old, cbba)

# ==============================
# VIDEO INIT (🔥 PERFECT FIX)
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

ret, frame = cap.read()
if not ret:
    print("Error reading video")
    exit()

# Detect rotation ONCE
rotate_flag = False
h, w = frame.shape[:2]

if h > w * 1.3:
    rotate_flag = True
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

# AFTER rotation → correct final size
h, w = frame.shape[:2]

fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)   # ✅ CORRECT SIZE
)

# restart video
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# ==============================
# MAIN LOOP
# ==============================

while True:

    ret, frame = cap.read()
    if not ret:
        break

    # apply SAME rotation
    if rotate_flag:
        frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

    results = yolo(frame, classes=VEHICLE_CLASSES, verbose=False)

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes.xyxy.cpu().numpy()

        for box in boxes:

            x1, y1, x2, y2 = map(int, box)
            center = ((x1 + x2)//2, (y1 + y2)//2)

            vid = assign_id(center)

            # perspective correction
            width_box = (x2 - x1)
            height_box = (y2 - y1)
            center_y = (y1 + y2) / 2

            area = (width_box * height_box) / ((center_y + 1) ** 1.2)

            if vid not in tracks:
                tracks[vid] = []

            tracks[vid].append(area)

            if len(tracks[vid]) > SEQ_LEN:
                tracks[vid] = tracks[vid][-SEQ_LEN:]

            # predict once
            if len(tracks[vid]) == SEQ_LEN and vid not in predicted:

                areas = np.array(tracks[vid])

                valid = np.where(~np.isnan(areas))[0]
                if len(valid) < 10:
                    continue

                areas = areas[valid[0]:valid[-1]+1]

                nans = np.isnan(areas)
                if np.any(nans):
                    x = np.arange(len(areas))
                    areas[nans] = np.interp(x[nans], x[~nans], areas[~nans])

                cbba = resize_cbba(areas)
                cbba = cbba / max_cbba

                tensor = torch.tensor(cbba).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

                with torch.no_grad():
                    pred = model(tensor).item()

                speed = pred * std_speed + mean_speed
                speed = 0.85 * speed - 1.5
                speed = max(0, speed)

                speeds[vid] = speed
                predicted.add(vid)

            # draw
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

            if vid in speeds:
                cv2.putText(
                    frame,
                    f"ID {vid}: {speeds[vid]:.2f} km/h",
                    (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0,0,255),
                    2
                )

    # 🔥 write EXACT frame (no resize)
    out.write(frame)

    cv2.imshow("FINAL OUTPUT (NO ZOOM FIXED)", frame)

    if cv2.waitKey(int(1000/fps)) & 0xFF == 27:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Output saved at:", OUTPUT_PATH)

Device: cuda
Model loaded successfully
Output saved at: C:\Users\natra\Downloads\top_veiw.mp4
